# Pratica 13 — Test A/B e disegno degli esperimenti

**Corso:** Statistica Applicata con Python · **Blocco 4:** Modelli ed esperimenti · **Tempo stimato:** 60 minuti
**Lezione associata:** `notebooks_clase/clase13_ab_testing_diseno_experimentos_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una cella con `# TODO`: completala.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere. Se dimentichi di completare un `TODO`, il messaggio te lo ricorda.
- Gli esercizi con simulazione usano tolleranze basate sull'errore standard, quindi un risultato corretto non dovrebbe fallire per caso.
- Le **soluzioni** sono alla fine del notebook, in una sezione a parte. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Tutti i dati sono sintetici: *esempi illustrativi*.

---
## Cella 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize, confint_proportions_2indep
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

print("Librerie e funzioni caricate correttamente.")

## Esercizio 1 — Dimensione campionaria, a mano e con statsmodels

Un negozio con conversione dell'8 % vuole rilevare un miglioramento al 10 % (MDE di +2 punti) con α = 0,05 e potenza del 90 %. Calcola: `za` e `zb` (quantili normali), `n_mano` (formula a mano, senza arrotondare), `h` (dimensione dell'effetto di `proportion_effectsize`, con il valore obiettivo per primo), `n_sm` (con `NormalIndPower().solve_power`), `n_grupo` (`n_sm` arrotondato per eccesso, intero), `dias` (giorni necessari con 1 000 visitatori al giorno in totale, ripartiti tra i due gruppi, arrotondato per eccesso) e `semanas` (`dias` in settimane complete).

In [ ]:
p_base, p_obj = 0.08, 0.10
alfa, potencia = 0.05, 0.90
visitantes_dia = 1000

za = None        # TODO: quantile z(1 − α/2)
zb = None        # TODO: quantile z(potenza)
n_mano = None    # TODO: (za + zb)² · [p1(1−p1) + p2(1−p2)] / (p2 − p1)²
h = None         # TODO
n_sm = None      # TODO
n_grupo = None   # TODO
dias = None      # TODO
semanas = None   # TODO

# Controllo automatico
assert all(v is not None for v in (za, zb, n_mano, h, n_sm, n_grupo, dias, semanas)), "Completa le righe contrassegnate con TODO"
assert np.isclose(za, 1.959964, atol=1e-4) and np.isclose(zb, 1.281552, atol=1e-4), "za = norm.ppf(1 - alfa/2); zb = norm.ppf(potenza)"
ref_mano = (za + zb) ** 2 * (0.08 * 0.92 + 0.10 * 0.90) / 0.02 ** 2
assert np.isclose(n_mano, ref_mano), "rivedi la formula: (za + zb)² · somma delle varianze / differenza²"
assert np.isclose(h, proportion_effectsize(0.10, 0.08)), "proportion_effectsize(p_obj, p_base)"
assert abs(n_sm - n_mano) < 0.01 * n_mano, "n_sm e n_mano dovrebbero differire meno dell'1 %"
assert n_grupo == int(np.ceil(NormalIndPower().solve_power(effect_size=proportion_effectsize(0.10, 0.08), alpha=0.05, power=0.90, ratio=1))), "n_grupo = ceil(n_sm)"
assert dias == int(np.ceil(2 * n_grupo / 1000)) and semanas == int(np.ceil(dias / 7)), "giorni = ceil(2·n_grupo / visitantes_dia); settimane = ceil(giorni / 7)"
print(f"n a mano: {n_mano:.1f}   n statsmodels: {n_sm:.1f}   per gruppo: {n_grupo}   giorni: {dias}   settimane: {semanas}")
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Analizzare un test A/B: p-value, IC e lift

Risultato di un esperimento: A ha convertito 300 visitatori su 5 000 e B ne ha convertiti 370 su 5 000. Calcola: `dif` (tasso di B meno tasso di A, come proporzione), `p_valor` (test per due proporzioni con `proportions_ztest`, mettendo B per primo), `ic_inf` e `ic_sup` (IC 95 % **a mano**, con l'errore standard non combinato `√(pA(1−pA)/n + pB(1−pB)/n)` e 1,96), `lift` (miglioramento relativo di B rispetto ad A, come proporzione), `significativo` (`True` se p < 0,05) e `incluye_bajo_mde` (`True` se l'IC contiene valori sotto l'effetto minimo rilevante di 0,01, cioè se `ic_inf < 0.01`).

In [ ]:
n_a = n_b = 5000
conv_a, conv_b = 300, 370

dif = None               # TODO
p_valor = None           # TODO
ic_inf = None            # TODO
ic_sup = None            # TODO
lift = None              # TODO
significativo = None     # TODO
incluye_bajo_mde = None  # TODO

# Controllo automatico
assert all(v is not None for v in (dif, p_valor, ic_inf, ic_sup, lift, significativo, incluye_bajo_mde)), "Completa le righe contrassegnate con TODO"
assert np.isclose(dif, 0.014), "dif = 370/5000 − 300/5000"
assert np.isclose(p_valor, proportions_ztest([370, 300], [5000, 5000])[1]), "proportions_ztest([conv_b, conv_a], [n_b, n_a]) e prendi il secondo valore"
ee_ref = np.sqrt(0.06 * 0.94 / 5000 + 0.074 * 0.926 / 5000)
assert np.isclose(ic_inf, 0.014 - 1.96 * ee_ref) and np.isclose(ic_sup, 0.014 + 1.96 * ee_ref), "IC = dif ± 1,96 · errore standard non combinato"
assert np.isclose(lift, 370 / 300 - 1), "lift = pB / pA − 1"
assert significativo == True and incluye_bajo_mde == True, "è significativo (p ≈ 0,005) ma l'IC scende sotto 0,01"
print(f"Differenza: {dif * 100:.2f} pp  p-value = {p_valor:.4f}  IC 95 % = [{ic_inf * 100:.2f}, {ic_sup * 100:.2f}] pp  lift = {lift * 100:.1f} %")
print("Esercizio 2 corretto ✔")

## Esercizio 3 — Simulare il peeking

Simuliamo 2 000 esperimenti A/A (stesso tasso reale, 10 %) con 5 controlli da 768 visitatori nuovi per gruppo ciascuno. La matrice `z` è già calcolata (2 000 righe, una per esperimento; 5 colonne, una per controllo). Completa: `sig` (matrice booleana: |z| > 1,96), `tasa_peeking` (proporzione di esperimenti significativi in **almeno un** controllo) e `tasa_final` (proporzione significativa guardando **solo** l'ultimo controllo).

In [ ]:
rng_e3 = np.random.default_rng(21)
nsim, controles, por = 2000, 5, 768
xa = rng_e3.binomial(por, 0.10, (nsim, controles)).cumsum(axis=1)
xb = rng_e3.binomial(por, 0.10, (nsim, controles)).cumsum(axis=1)
ns = por * np.arange(1, controles + 1)
pp = (xa + xb) / (2 * ns)
z = ((xb - xa) / ns) / np.sqrt(pp * (1 - pp) * 2 / ns)

sig = None            # TODO: matrice booleana |z| > 1,96
tasa_peeking = None   # TODO
tasa_final = None     # TODO

# Controllo automatico
assert all(v is not None for v in (sig, tasa_peeking, tasa_final)), "Completa le righe contrassegnate con TODO"
assert sig.shape == (2000, 5) and sig.dtype == bool, "sig = np.abs(z) > 1.96 (matrice booleana 2000 × 5)"
assert np.isclose(tasa_peeking, (np.abs(z) > 1.96).any(axis=1).mean()), "usa sig.any(axis=1).mean(): significativo in almeno un controllo"
assert np.isclose(tasa_final, (np.abs(z[:, -1]) > 1.96).mean()), "usa sig[:, -1].mean(): solo l'ultimo controllo"
# Atteso: ≈ 5 % guardando una volta (± 3 errori standard ≈ 1,5 pp) e ≈ 13 % con 5 controlli
assert 0.035 < tasa_final < 0.065, "guardando solo alla fine il tasso dovrebbe aggirarsi intorno al 5 %"
assert 0.10 < tasa_peeking < 0.17, "con 5 controlli il tasso reale di falsi positivi dovrebbe aggirarsi intorno al 13 %"
print(f"Falsi positivi guardando una volta: {tasa_final * 100:.1f} %   guardando 5 volte: {tasa_peeking * 100:.1f} %")
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Controllo SRM con chi-quadro

Due esperimenti disegnati con ripartizione 50/50 registrano queste dimensioni: esperimento 1, A = 25 300 e B = 24 200; esperimento 2, A = 24 900 e B = 24 600. Per ciascuno calcola il chi-quadro di bontà di adattamento con `stats.chisquare` (dimensioni attese uguali): `chi2_1`, `p_1`, `chi2_2`, `p_2`. Poi `srm_1` e `srm_2` valgono `True` se il p-value è minore di 0,001. Infine `chi2_1_mano` calcola il chi-quadro dell'esperimento 1 **a mano**: la somma di (osservato − atteso)² / atteso.

In [ ]:
obs_1 = np.array([25300, 24200])
obs_2 = np.array([24900, 24600])

chi2_1 = None       # TODO
p_1 = None          # TODO
chi2_2 = None       # TODO
p_2 = None          # TODO
srm_1 = None        # TODO
srm_2 = None        # TODO
chi2_1_mano = None  # TODO

# Controllo automatico
assert all(v is not None for v in (chi2_1, p_1, chi2_2, p_2, srm_1, srm_2, chi2_1_mano)), "Completa le righe contrassegnate con TODO"
assert np.isclose(chi2_1, 24.4444, atol=1e-3) and np.isclose(chi2_2, 1.8182, atol=1e-3), "stats.chisquare(osservati) restituisce (chi2, p); l'atteso di default è la ripartizione uguale"
assert srm_1 == True and srm_2 == False, "l'esperimento 1 ha SRM (p ≈ 0,0000008) e il 2 no (p ≈ 0,18)"
assert np.isclose(chi2_1_mano, chi2_1), "atteso = totale / 2 = 24 750; somma di (oss − att)² / att per i due gruppi"
print(f"Esperimento 1: chi2 = {chi2_1:.2f}  p = {p_1:.7f}  SRM = {srm_1}   |   Esperimento 2: chi2 = {chi2_2:.2f}  p = {p_2:.3f}  SRM = {srm_2}")
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Il paradosso di Simpson

Un esperimento con due tipi di utenti (`nuevos` e `recurrentes`). Con la tabella `tabla`, calcola: `tasa_seg` (DataFrame con il tasso di conversione di ciascun segmento —righe— e gruppo —colonne—), `tasa_total` (Serie con il tasso totale di ciascun gruppo), `simpson` (`True` se B vince in **tutti** i segmenti e perde nel totale) e `dif_estand` (differenza B − A, in punti percentuali, usando come peso di ciascun segmento la sua proporzione sul totale degli utenti di A e B insieme).

In [ ]:
tabla = pd.DataFrame({"segmento": ["nuevos", "nuevos", "recurrentes", "recurrentes"],
                      "grupo": ["A", "B", "A", "B"],
                      "usuarios": [200, 700, 800, 300],
                      "conversiones": [20, 84, 240, 96]})
conv = tabla.pivot(index="segmento", columns="grupo", values="conversiones")
usr = tabla.pivot(index="segmento", columns="grupo", values="usuarios")

tasa_seg = None     # TODO
tasa_total = None   # TODO
simpson = None      # TODO
dif_estand = None   # TODO

# Controllo automatico
assert all(v is not None for v in (tasa_seg, tasa_total, simpson, dif_estand)), "Completa le righe contrassegnate con TODO"
assert np.isclose(tasa_seg.loc["nuevos", "A"], 0.10) and np.isclose(tasa_seg.loc["recurrentes", "B"], 0.32), "tasa_seg = conv / usr (segmenti in righe, gruppi in colonne)"
assert np.isclose(tasa_total["A"], 0.26) and np.isclose(tasa_total["B"], 0.18), "tasso totale = conv.sum() / usr.sum()"
assert simpson == True, "B vince nei due segmenti (12 % > 10 %, 32 % > 30 %) e perde nel totale (18 % < 26 %)"
assert np.isclose(dif_estand, 2.0), "pesi = utenti del segmento (A + B) / totale; tasso standardizzato = somma(peso · tasso del segmento)"
print(tasa_seg.round(3).to_string())
print("Totale:", (tasa_total * 100).round(1).to_dict(), "  differenza standardizzata (pp):", round(dif_estand, 2))
print("Esercizio 5 corretto ✔")

---
## Sfida finale — Report di un esperimento A/B

Un negozio pianifica un test A/B: conversione attuale dell'8 %, miglioramento minimo da rilevare di +1,5 punti (fino al 9,5 %), α = 0,05, potenza dell'80 %. L'esperimento è già stato eseguito e i dati sono in `datos_reto` (colonne `grupo` e `convierte`). Elabora il report: `n_plan` (dimensione per gruppo pianificata, con `NormalIndPower` e `proportion_effectsize`, arrotondata per eccesso), `tamanos` (dizionario con gli utenti reali di `A` e `B`), `p_srm` (p-value del chi-quadro SRM su quelle dimensioni, ripartizione attesa 50/50), `p_valor` (test per due proporzioni, B per primo), `ic_inf` e `ic_sup` (IC 95 % della differenza B − A, con `confint_proportions_2indep` metodo `"newcomb"`) e `decision`: `"SRM"` se c'è SRM (p_srm < 0,001); altrimenti `"GO"` (lanciare B) se p_valor < 0,05 e `ic_inf` > 0, e `"NO_GO"` negli altri casi. Scrivi in `informe` un paragrafo breve (minimo 60 caratteri) con disegno, controlli, risultato e decisione.

In [ ]:
def generar_experimento(rng, n_a=5561, n_b=5548, p_a=0.08, p_b=0.095):
    """Dati sintetici (esempio illustrativo): una riga per visitatore."""
    convierte = np.concatenate([rng.random(n_a) < p_a, rng.random(n_b) < p_b]).astype(int)
    return pd.DataFrame({"grupo": ["A"] * n_a + ["B"] * n_b, "convierte": convierte})

datos_reto = generar_experimento(np.random.default_rng(95))

n_plan = None     # TODO
tamanos = None    # TODO
p_srm = None      # TODO
p_valor = None    # TODO
ic_inf = None     # TODO
ic_sup = None     # TODO
decision = None   # TODO
informe = None    # TODO

# Controllo automatico
assert all(v is not None for v in (n_plan, tamanos, p_srm, p_valor, ic_inf, ic_sup, decision, informe)), "Completa le righe contrassegnate con TODO"
ref_n = int(np.ceil(NormalIndPower().solve_power(effect_size=proportion_effectsize(0.095, 0.08), alpha=0.05, power=0.80, ratio=1)))
assert n_plan == ref_n, "n_plan = ceil(NormalIndPower().solve_power(proportion_effectsize(0.095, 0.08), alpha=0.05, power=0.80, ratio=1))"
g = datos_reto.groupby("grupo")["convierte"].agg(["sum", "count"])
assert tamanos == {"A": int(g.loc["A", "count"]), "B": int(g.loc["B", "count"])}, "tamanos = {'A': utenti di A, 'B': utenti di B}"
assert np.isclose(p_srm, stats.chisquare([g.loc["A", "count"], g.loc["B", "count"]])[1]), "stats.chisquare sulle due dimensioni, ripartizione attesa uguale"
ref_p = proportions_ztest([g.loc["B", "sum"], g.loc["A", "sum"]], [g.loc["B", "count"], g.loc["A", "count"]])[1]
assert np.isclose(p_valor, ref_p), "proportions_ztest([conv_B, conv_A], [n_B, n_A]) e prendi il secondo valore"
ref_lo, ref_hi = confint_proportions_2indep(g.loc["B", "sum"], g.loc["B", "count"], g.loc["A", "sum"], g.loc["A", "count"], method="newcomb")
assert np.isclose(ic_inf, ref_lo) and np.isclose(ic_sup, ref_hi), "confint_proportions_2indep(conv_B, n_B, conv_A, n_A, method='newcomb')"
assert p_srm > 0.001 and p_valor < 0.05 and ic_inf > 0, "in questi dati: nessun SRM, significativo, e l'IC non include lo 0"
assert decision == "GO", "applica la regola: SRM → revisar; p < 0,05 e ic_inf > 0 → lanzar B; altrimenti non lanzare B"
assert isinstance(informe, str) and len(informe) >= 60, "scrivi un paragrafo di almeno 60 caratteri"
print(f"n_plan = {n_plan}   dimensioni: {tamanos}   p_srm = {p_srm:.3f}   p-value = {p_valor:.4f}   IC = [{ic_inf * 100:.2f}, {ic_sup * 100:.2f}] pp")
print("Decisione:", decision)
print("Sfida finale corretta ✔")

---
## Chiusura della pratica

Hai percorso il ciclo di un test A/B: calcolare la dimensione campionaria, analizzare con IC e lift, misurare l'effetto del *peeking*, verificare l'SRM e riconoscere il paradosso di Simpson. Prosegui con la lezione 14: Progetto integrativo.

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
p_base, p_obj = 0.08, 0.10
alfa, potencia = 0.05, 0.90
visitantes_dia = 1000

za = stats.norm.ppf(1 - alfa / 2)
zb = stats.norm.ppf(potencia)
n_mano = (za + zb) ** 2 * (p_base * (1 - p_base) + p_obj * (1 - p_obj)) / (p_obj - p_base) ** 2
h = proportion_effectsize(p_obj, p_base)
n_sm = NormalIndPower().solve_power(effect_size=h, alpha=alfa, power=potencia, ratio=1)
n_grupo = int(np.ceil(n_sm))
dias = int(np.ceil(2 * n_grupo / visitantes_dia))
semanas = int(np.ceil(dias / 7))

assert all(v is not None for v in (za, zb, n_mano, h, n_sm, n_grupo, dias, semanas)), "Completa le righe contrassegnate con TODO"
assert np.isclose(za, 1.959964, atol=1e-4) and np.isclose(zb, 1.281552, atol=1e-4), "za = norm.ppf(1 - alfa/2); zb = norm.ppf(potenza)"
ref_mano = (za + zb) ** 2 * (0.08 * 0.92 + 0.10 * 0.90) / 0.02 ** 2
assert np.isclose(n_mano, ref_mano), "rivedi la formula: (za + zb)² · somma delle varianze / differenza²"
assert np.isclose(h, proportion_effectsize(0.10, 0.08)), "proportion_effectsize(p_obj, p_base)"
assert abs(n_sm - n_mano) < 0.01 * n_mano, "n_sm e n_mano dovrebbero differire meno dell'1 %"
assert n_grupo == int(np.ceil(NormalIndPower().solve_power(effect_size=proportion_effectsize(0.10, 0.08), alpha=0.05, power=0.90, ratio=1))), "n_grupo = ceil(n_sm)"
assert dias == int(np.ceil(2 * n_grupo / 1000)) and semanas == int(np.ceil(dias / 7)), "giorni = ceil(2·n_grupo / visitantes_dia); settimane = ceil(giorni / 7)"
print(f"n a mano: {n_mano:.1f}   n statsmodels: {n_sm:.1f}   per gruppo: {n_grupo}   giorni: {dias}   settimane: {semanas}")
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
n_a = n_b = 5000
conv_a, conv_b = 300, 370

pa, pb = conv_a / n_a, conv_b / n_b
dif = pb - pa
p_valor = proportions_ztest([conv_b, conv_a], [n_b, n_a])[1]
ee = np.sqrt(pa * (1 - pa) / n_a + pb * (1 - pb) / n_b)
ic_inf, ic_sup = dif - 1.96 * ee, dif + 1.96 * ee
lift = pb / pa - 1
significativo = bool(p_valor < 0.05)
incluye_bajo_mde = bool(ic_inf < 0.01)

assert all(v is not None for v in (dif, p_valor, ic_inf, ic_sup, lift, significativo, incluye_bajo_mde)), "Completa le righe contrassegnate con TODO"
assert np.isclose(dif, 0.014), "dif = 370/5000 − 300/5000"
assert np.isclose(p_valor, proportions_ztest([370, 300], [5000, 5000])[1]), "proportions_ztest([conv_b, conv_a], [n_b, n_a]) e prendi il secondo valore"
ee_ref = np.sqrt(0.06 * 0.94 / 5000 + 0.074 * 0.926 / 5000)
assert np.isclose(ic_inf, 0.014 - 1.96 * ee_ref) and np.isclose(ic_sup, 0.014 + 1.96 * ee_ref), "IC = dif ± 1,96 · errore standard non combinato"
assert np.isclose(lift, 370 / 300 - 1), "lift = pB / pA − 1"
assert significativo == True and incluye_bajo_mde == True, "è significativo (p ≈ 0,005) ma l'IC scende sotto 0,01"
print(f"Differenza: {dif * 100:.2f} pp  p-value = {p_valor:.4f}  IC 95 % = [{ic_inf * 100:.2f}, {ic_sup * 100:.2f}] pp  lift = {lift * 100:.1f} %")
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
rng_e3 = np.random.default_rng(21)
nsim, controles, por = 2000, 5, 768
xa = rng_e3.binomial(por, 0.10, (nsim, controles)).cumsum(axis=1)
xb = rng_e3.binomial(por, 0.10, (nsim, controles)).cumsum(axis=1)
ns = por * np.arange(1, controles + 1)
pp = (xa + xb) / (2 * ns)
z = ((xb - xa) / ns) / np.sqrt(pp * (1 - pp) * 2 / ns)

sig = np.abs(z) > 1.96
tasa_peeking = sig.any(axis=1).mean()
tasa_final = sig[:, -1].mean()

assert all(v is not None for v in (sig, tasa_peeking, tasa_final)), "Completa le righe contrassegnate con TODO"
assert sig.shape == (2000, 5) and sig.dtype == bool, "sig = np.abs(z) > 1.96 (matrice booleana 2000 × 5)"
assert np.isclose(tasa_peeking, (np.abs(z) > 1.96).any(axis=1).mean()), "usa sig.any(axis=1).mean(): significativo in almeno un controllo"
assert np.isclose(tasa_final, (np.abs(z[:, -1]) > 1.96).mean()), "usa sig[:, -1].mean(): solo l'ultimo controllo"
# Atteso: ≈ 5 % guardando una volta (± 3 errori standard ≈ 1,5 pp) e ≈ 13 % con 5 controlli
assert 0.035 < tasa_final < 0.065, "guardando solo alla fine il tasso dovrebbe aggirarsi intorno al 5 %"
assert 0.10 < tasa_peeking < 0.17, "con 5 controlli il tasso reale di falsi positivi dovrebbe aggirarsi intorno al 13 %"
print(f"Falsi positivi guardando una volta: {tasa_final * 100:.1f} %   guardando 5 volte: {tasa_peeking * 100:.1f} %")
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
obs_1 = np.array([25300, 24200])
obs_2 = np.array([24900, 24600])

chi2_1, p_1 = stats.chisquare(obs_1)
chi2_2, p_2 = stats.chisquare(obs_2)
srm_1 = bool(p_1 < 0.001)
srm_2 = bool(p_2 < 0.001)
esp_1 = obs_1.sum() / 2
chi2_1_mano = float(((obs_1 - esp_1) ** 2 / esp_1).sum())

assert all(v is not None for v in (chi2_1, p_1, chi2_2, p_2, srm_1, srm_2, chi2_1_mano)), "Completa le righe contrassegnate con TODO"
assert np.isclose(chi2_1, 24.4444, atol=1e-3) and np.isclose(chi2_2, 1.8182, atol=1e-3), "stats.chisquare(osservati) restituisce (chi2, p); l'atteso di default è la ripartizione uguale"
assert srm_1 == True and srm_2 == False, "l'esperimento 1 ha SRM (p ≈ 0,0000008) e il 2 no (p ≈ 0,18)"
assert np.isclose(chi2_1_mano, chi2_1), "atteso = totale / 2 = 24 750; somma di (oss − att)² / att per i due gruppi"
print(f"Esperimento 1: chi2 = {chi2_1:.2f}  p = {p_1:.7f}  SRM = {srm_1}   |   Esperimento 2: chi2 = {chi2_2:.2f}  p = {p_2:.3f}  SRM = {srm_2}")
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
tabla = pd.DataFrame({"segmento": ["nuevos", "nuevos", "recurrentes", "recurrentes"],
                      "grupo": ["A", "B", "A", "B"],
                      "usuarios": [200, 700, 800, 300],
                      "conversiones": [20, 84, 240, 96]})
conv = tabla.pivot(index="segmento", columns="grupo", values="conversiones")
usr = tabla.pivot(index="segmento", columns="grupo", values="usuarios")

tasa_seg = conv / usr
tasa_total = conv.sum() / usr.sum()
simpson = bool((tasa_seg["B"] > tasa_seg["A"]).all() and tasa_total["B"] < tasa_total["A"])
pesos = usr.sum(axis=1) / usr.values.sum()
estand = tasa_seg.mul(pesos, axis=0).sum()
dif_estand = float((estand["B"] - estand["A"]) * 100)

assert all(v is not None for v in (tasa_seg, tasa_total, simpson, dif_estand)), "Completa le righe contrassegnate con TODO"
assert np.isclose(tasa_seg.loc["nuevos", "A"], 0.10) and np.isclose(tasa_seg.loc["recurrentes", "B"], 0.32), "tasa_seg = conv / usr (segmenti in righe, gruppi in colonne)"
assert np.isclose(tasa_total["A"], 0.26) and np.isclose(tasa_total["B"], 0.18), "tasso totale = conv.sum() / usr.sum()"
assert simpson == True, "B vince nei due segmenti (12 % > 10 %, 32 % > 30 %) e perde nel totale (18 % < 26 %)"
assert np.isclose(dif_estand, 2.0), "pesi = utenti del segmento (A + B) / totale; tasso standardizzato = somma(peso · tasso del segmento)"
print(tasa_seg.round(3).to_string())
print("Totale:", (tasa_total * 100).round(1).to_dict(), "  differenza standardizzata (pp):", round(dif_estand, 2))
print("Esercizio 5 corretto ✔")

### Soluzione della sfida finale

In [ ]:
def generar_experimento(rng, n_a=5561, n_b=5548, p_a=0.08, p_b=0.095):
    """Dati sintetici (esempio illustrativo): una riga per visitatore."""
    convierte = np.concatenate([rng.random(n_a) < p_a, rng.random(n_b) < p_b]).astype(int)
    return pd.DataFrame({"grupo": ["A"] * n_a + ["B"] * n_b, "convierte": convierte})

datos_reto = generar_experimento(np.random.default_rng(95))

n_plan = int(np.ceil(NormalIndPower().solve_power(effect_size=proportion_effectsize(0.095, 0.08), alpha=0.05, power=0.80, ratio=1)))
g = datos_reto.groupby("grupo")["convierte"].agg(["sum", "count"])
tamanos = {"A": int(g.loc["A", "count"]), "B": int(g.loc["B", "count"])}
p_srm = stats.chisquare([tamanos["A"], tamanos["B"]])[1]
p_valor = proportions_ztest([g.loc["B", "sum"], g.loc["A", "sum"]], [tamanos["B"], tamanos["A"]])[1]
ic_inf, ic_sup = confint_proportions_2indep(g.loc["B", "sum"], tamanos["B"], g.loc["A", "sum"], tamanos["A"], method="newcomb")
if p_srm < 0.001:
    decision = "SRM"
elif p_valor < 0.05 and ic_inf > 0:
    decision = "GO"
else:
    decision = "NO_GO"
informe = "Disegno: n = 5 561 per gruppo per rilevare +1,5 pp (8 % → 9,5 %) con α = 5 % e potenza 80 %. Controlli: nessun SRM. Risultato: B supera A con un IC che non include lo 0. Decisione: lanciare B, sapendo che l'effetto plausibile è modesto."

assert all(v is not None for v in (n_plan, tamanos, p_srm, p_valor, ic_inf, ic_sup, decision, informe)), "Completa le righe contrassegnate con TODO"
ref_n = int(np.ceil(NormalIndPower().solve_power(effect_size=proportion_effectsize(0.095, 0.08), alpha=0.05, power=0.80, ratio=1)))
assert n_plan == ref_n, "n_plan = ceil(NormalIndPower().solve_power(proportion_effectsize(0.095, 0.08), alpha=0.05, power=0.80, ratio=1))"
g = datos_reto.groupby("grupo")["convierte"].agg(["sum", "count"])
assert tamanos == {"A": int(g.loc["A", "count"]), "B": int(g.loc["B", "count"])}, "tamanos = {'A': utenti di A, 'B': utenti di B}"
assert np.isclose(p_srm, stats.chisquare([g.loc["A", "count"], g.loc["B", "count"]])[1]), "stats.chisquare sulle due dimensioni, ripartizione attesa uguale"
ref_p = proportions_ztest([g.loc["B", "sum"], g.loc["A", "sum"]], [g.loc["B", "count"], g.loc["A", "count"]])[1]
assert np.isclose(p_valor, ref_p), "proportions_ztest([conv_B, conv_A], [n_B, n_A]) e prendi il secondo valore"
ref_lo, ref_hi = confint_proportions_2indep(g.loc["B", "sum"], g.loc["B", "count"], g.loc["A", "sum"], g.loc["A", "count"], method="newcomb")
assert np.isclose(ic_inf, ref_lo) and np.isclose(ic_sup, ref_hi), "confint_proportions_2indep(conv_B, n_B, conv_A, n_A, method='newcomb')"
assert p_srm > 0.001 and p_valor < 0.05 and ic_inf > 0, "in questi dati: nessun SRM, significativo, e l'IC non include lo 0"
assert decision == "GO", "applica la regola: SRM → revisar; p < 0,05 e ic_inf > 0 → lanzar B; altrimenti non lanzare B"
assert isinstance(informe, str) and len(informe) >= 60, "scrivi un paragrafo di almeno 60 caratteri"
print(f"n_plan = {n_plan}   dimensioni: {tamanos}   p_srm = {p_srm:.3f}   p-value = {p_valor:.4f}   IC = [{ic_inf * 100:.2f}, {ic_sup * 100:.2f}] pp")
print("Decisione:", decision)
print("Sfida finale corretta ✔")

---
## Autovalutazione

Annota in una cella di testo o su carta:

- **Quali esercizi ho risolto senza aiuto?**
- **Quale concetto resta poco chiaro?** (ad esempio: formula di n, potenza, peeking, SRM, paradosso di Simpson)
- **Tempo dedicato:** ___ minuti
- **Una domanda che voglio portare alla prossima lezione:** ...